# Exposure-conditioned repeated-structure count atlas

**Projection pilot, not yet a full latent-count estimator.** Repeated PT sections share a canonical ordered anatomical object. Unknown nephron identities and nephron-level variation remain integrated out; specimens, not sections or thinning repeats, are biological replication units.

Notebook 16 reconstructs a shared weakly anchored axis without DPT using count-residual PCs. Its Gaussian projection is more robust than log-normalized projection but still moves under severe read thinning. Here that fitted **training-only coordinate** initializes a canonical gene-rate path. At each position, smooth count/exposure rates are estimated separately in each reference specimen and averaged equally, preventing specimen occupancy from defining the canonical molecular mean.

\[
C_{s,i,g}\mid z,L_i\sim\mathrm{NB}(L_i p_g(z),\theta=100).
\]

Project raw counts using this conditional observation likelihood, then test the same held-out sections under count thinning. A shared rate path remains a special case of the repeated-structure distributional atlas. This pilot has no explicit specimen intercept in its count density; initialization uses the restricted offsets from notebook 16. Fixed dispersion and independent genes are strong approximations, so posterior precision is **not calibrated anatomical uncertainty**. DPT and tissue depth are neither required nor used. Winding anatomy and cuts from different nephrons make depth an imperfect supporting proxy, not a method-selection target.

The decisive evidence here is frozen cross-specimen molecular prediction, within-segment robustness and comparison with strong coarse/PC1 alternatives. Ordinal training labels still inherit marker information. Read thinning preserves section identity but does not simulate changing cell composition or segmentation area.

In [ ]:
from pathlib import Path
import argparse, hashlib, io, json, os, sys
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.stats import spearmanr
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pseudospace').is_dir())
sys.path.insert(0, str(REPO))
from pseudospace.repeated_inputs import load_repeated_mouse_inputs, _read_and_validate_segmentation
from pseudospace.joint_repeated_atlas import fit_joint_atlas, project_joint_atlas, fit_balanced_representation, transform_balanced_representation
from pseudospace.repeated_structure import fit_atlas
from pseudospace.canonical_mean_atlas import segment_means
from pseudospace.spatial_probability_flow import scaffold_centroid_rows
from pseudospace.stage_cache import cached_payload
parser=argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', default=os.environ.get('PSEUDOSPACE_DATA_ROOT', str(REPO/'data')))
parser.add_argument('--results-root', default=os.environ.get('PSEUDOSPACE_RESULTS_ROOT', str(REPO/'results')))
args,_=parser.parse_known_args()
DATA_ROOT,RESULTS_ROOT=Path(args.data_root).expanduser(),Path(args.results_root).expanduser()
OUT=RESULTS_ROOT/'count_rate_projection_pilot'
OUT.mkdir(parents=True,exist_ok=True)
CONTROLS=('Ctrl1A2','Ctrl1A4')
NOTEBOOK_LOGIC_VERSION='17.count_rate_projection.1'
SEED,N_COMPONENTS,N_FOLDS=15,15,3
GRID=np.linspace(0,1,101)
MODEL_PARAMS=dict(grid=GRID,roughness=.001,offset_penalty=4.,anatomy_strength=1.,anatomy_width=.1,max_iter=60,tol=1e-5)
MARKERS=set('Slc5a2 Slc5a12 Gatm Slc22a6 Slc13a3 Slc1a1 Cyp2e1 Slc7a13 Slc22a7 Slc5a1 Cyp7b1 Slc7a12 Lrp2 Cubn Slc34a1 Slc6a18 Acsm3 Agt'.casefold().split())
CODE_FILES=[REPO/'pseudospace'/f for f in ['joint_repeated_atlas.py','repeated_structure.py','repeated_inputs.py','canonical_mean_atlas.py','count_representation.py','count_rate_atlas.py','repeated_atlas_baselines.py']]
CODE_DIGEST=hashlib.sha256(b''.join(p.read_bytes() for p in CODE_FILES)).hexdigest()
CACHE_PARAMS=dict(version=NOTEBOOK_LOGIC_VERSION,seed=SEED,components=N_COMPONENTS,folds=N_FOLDS,model={**MODEL_PARAMS,'grid':GRID.tolist()},gene_prediction_ridge=.001)
plt.rcParams.update({'font.size':9,'axes.spines.top':False,'axes.spines.right':False})
from pseudospace.count_representation import fit_count_representation, transform_count_representation
from pseudospace.count_rate_atlas import fit_count_rate_atlas, project_count_rate_atlas
from pseudospace.repeated_atlas_baselines import fit_pc1_reference, project_pc1_reference, fit_segment_mixture, predict_segment_mixture
RATE_PARAMS=dict(grid=GRID,bandwidth=.08,theta=100.,min_effective=12.)
CACHE_PARAMS.update(rate_bandwidth=.08,theta=100.,minimum_effective=12.,thinning_fractions=[.75,.5,.25],thinning_repeats=2,classifier_C=1.)

## Validated healthy inputs and frozen splits

Use exactly the deterministic profile split, fold-0 gene split and marker exclusions from notebook 16. All healthy PT sections remain eligible; absence from a DPT export cannot exclude one. Count exposure sums all measured genes, including excluded/withheld genes, retaining the disclosed compositional dependence of gene holdouts. Outcomes never choose the dispersion or bandwidth in this pilot.

In [ ]:
healthy=load_repeated_mouse_inputs(DATA_ROOT,samples=CONTROLS)
obs=healthy.obs.copy()
ids=obs.structure_id.to_numpy(str)
specimens=obs['sample'].astype(str).to_numpy()
labels=obs.segment_class.astype(str).str.replace('PT-','',regex=False).to_numpy()
if not np.isin(labels,['S1','S2','S3']).all():
    raise ValueError('Resolve uncertain PT labels upstream')
anatomy=np.array([{'S1':0,'S2':1,'S3':2}[a] for a in labels])
genes=healthy.var_names.to_numpy(str)
M=healthy.layers['lognorm']
X=M.toarray().astype(float) if sparse.issparse(M) else np.asarray(M,float)
if not np.isfinite(X).all():
    raise ValueError('Nonfinite expression')
marker_mask=np.array([g.casefold() in MARKERS for g in genes])
gene_fold=np.array([int(hashlib.sha256(g.casefold().encode()).hexdigest()[:8],16)%N_FOLDS for g in genes])
validation=np.array([int(hashlib.sha256(s.encode()).hexdigest()[:8],16)%5==0 for s in ids])
for s in CONTROLS:
    if min(np.sum((specimens==s)&validation),np.sum((specimens==s)&~validation))<30:
        raise ValueError('Insufficient fixed holdout support')
manifest=dict(healthy.uns['repeated_mouse_input_manifest'])
manifest.update(notebook_logic_version=NOTEBOOK_LOGIC_VERSION,config=CACHE_PARAMS,code_digest=CODE_DIGEST,markers_excluded=sorted(MARKERS),validation_profiles=ids[validation].tolist(),dpt_used_for_fitting=False)
(OUT/'input_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.crosstab([specimens,validation],labels))
print('Profiles:',len(X),'genes:',len(genes),'dense GiB:',round(X.nbytes/2**30,3))
raw_counts=healthy.layers['counts']
raw_total=np.asarray(raw_counts.sum(axis=1)).ravel()

In [ ]:
def initial_coordinate(Y,a,kind,seed):
    if kind=='pca':
        value=Y[:,0].copy()
        if np.median(value[a==2])<np.median(value[a==0]):
            value=-value
        lo,hi=np.quantile(value,[.01,.99])
        if hi<=lo:
            raise ValueError('Degenerate PCA initializer')
        return np.clip((value-lo)/(hi-lo),0,1)
    if kind=='ordinal_random':
        rng=np.random.default_rng(seed)
        return np.array([rng.beta(*[(2,6),(3,3),(6,2)][int(k)]) for k in a])
    raise ValueError(kind)

def gene_predictions(train_X,train_z,train_s,query_z):
    atlas=fit_atlas(train_X,train_z,train_s,grid=GRID,ridge=.001)
    return atlas.predict(query_z)

def score_rows(tr,te,eg,pred,key):
    rows=[]
    for s in np.unique(specimens[te]):
        take=specimens[te]==s
        for segment in ['all','S1','S2','S3']:
            use=take if segment=='all' else take&(labels[te]==segment)
            if not use.any():
                continue
            reference=X[tr][:,eg]
            mean=np.mean([reference[specimens[tr]==q].mean(axis=0) for q in np.unique(specimens[tr])],axis=0)
            variance=np.maximum(np.mean([((reference[specimens[tr]==q]-mean)**2).mean(axis=0) for q in np.unique(specimens[tr])],axis=0),1e-3)
            error=np.mean((X[te][:,eg][use]-pred[use])**2/variance,axis=0)
            rows.append(dict(**key,test_specimen=s,segment=segment,n=int(use.sum()),gene_mse=float(error.mean())))
    return rows



def dense_panel(matrix,rows,columns):
    value=matrix[rows][:,columns]
    return value.toarray() if sparse.issparse(value) else np.asarray(value)

def gaussian_projection(atlas,Y,query_specimens):
    out={k:np.empty(len(Y)) for k in ['z_mean','z_MAP','entropy']}
    for s in np.unique(query_specimens):
        use=query_specimens==s
        pp=project_joint_atlas(atlas,Y[use],offset=atlas['offsets'].get(s,np.zeros(Y.shape[1])))
        for k in out:
            out[k][use]=pp[k]
    return out

def thin_counts(matrix,fraction,seed):
    if not np.isfinite(fraction) or not 0<fraction<=1:
        raise ValueError('Thinning fraction must lie in (0,1]')
    rng=np.random.default_rng(seed)
    if sparse.issparse(matrix):
        value=matrix.copy().tocsr()
        value.data=rng.binomial(np.rint(value.data).astype(np.int64),fraction)
        value.eliminate_zeros()
        return value
    return rng.binomial(np.rint(matrix).astype(np.int64),fraction)

probe=sparse.csr_matrix([[5,0,12],[0,9,4]])
assert np.array_equal(thin_counts(probe,1.,SEED).toarray(),probe.toarray())
assert np.array_equal(thin_counts(probe,.5,SEED).toarray(),thin_counts(probe,.5,SEED).toarray())

CACHE_INPUTS=dict(X=X,raw_counts=raw_counts,library=raw_total,ids=ids,specimens=specimens,labels_training_and_evaluation=labels,gene_fold=gene_fold,marker_mask=marker_mask,validation=validation)

## Frozen projections and withheld-gene prediction

Fit one joint/profile-holdout reference and two one-control references, using modeling genes only. Query labels enter only the explicit segment oracle and descriptive evaluation. A new specimen gets zero Gaussian offset; the count atlas has one shared reference rate path. No query calibration or DPT occurs.

Conditional log densities are comparable only under the same count model/exposure assumptions; they are not anatomical accuracy measures. Gene predictions use the same log-expression response and shared spline protocol as notebook 16, preserving a comparable response metric. Independent count genes may yield overconfident posteriors because real genes share programs.

In [ ]:
def run_rate_projection():
    mg=np.flatnonzero(~marker_mask&(gene_fold!=0))
    eg=np.flatnonzero(~marker_mask&(gene_fold==0))
    designs=[(~validation,validation,'joint_profile_holdout')]
    designs.extend((specimens==s,specimens!=s,'heldout_specimen_'+s) for s in CONTROLS)
    scores,coordinates,thinning_rows,diagnostics,failures=[],[],[],[],[]
    for tr,te,protocol in designs:
        try:
            reference=dense_panel(raw_counts,tr,mg)
            query_counts=dense_panel(raw_counts,te,mg)
            Y,transform=fit_count_representation(reference,raw_total[tr],specimens[tr],N_COMPONENTS,theta=100.,seed=SEED)
            query=transform_count_representation(query_counts,raw_total[te],transform)
            gaussian=fit_joint_atlas(Y,specimens[tr],anatomy[tr],initial_coordinate(Y,anatomy[tr],'pca',SEED),**MODEL_PARAMS)
            ga_train=gaussian_projection(gaussian,Y,specimens[tr])
            ga_query=gaussian_projection(gaussian,query,specimens[te])
            rate=fit_count_rate_atlas(reference,raw_total[tr],ga_train['z_mean'],specimens[tr],**RATE_PARAMS)
            print('Fitted rates:',protocol,'minimum support:',rate['n_effective'].min(),flush=True)
            nb_train=project_count_rate_atlas(rate,reference,raw_total[tr])
            nb_query=project_count_rate_atlas(rate,query_counts,raw_total[te])
            key=dict(protocol=protocol,fold=0,initializer='pca',seed=SEED,variant='count_rate_pilot')
            for method,train_position,projected in [('count_gaussian',ga_train['z_mean'],ga_query),('count_NB',nb_train['z_mean'],nb_query)]:
                pred=gene_predictions(X[tr][:,eg],train_position,specimens[tr],projected['z_mean'])
                scores.extend(score_rows(tr,te,eg,pred,{**key,'predictor':method}))
                coordinates.extend(dict(**key,method=method,structure_id=sid,test_specimen=s,z=float(zz),z_MAP=float(zmap),entropy=float(entropy)) for sid,s,zz,zmap,entropy in zip(ids[te],specimens[te],projected['z_mean'],projected['z_MAP'],projected['entropy']))
            pc1=fit_pc1_reference(Y,anatomy[tr])
            pc_pred=gene_predictions(X[tr][:,eg],project_pc1_reference(pc1,Y),specimens[tr],project_pc1_reference(pc1,query))
            scores.extend(score_rows(tr,te,eg,pc_pred,{**key,'predictor':'frozen_count_PC1'}))
            mixture=fit_segment_mixture(Y,X[tr][:,eg],anatomy[tr],specimens[tr],C=1.,seed=SEED)
            scores.extend(score_rows(tr,te,eg,predict_segment_mixture(mixture,query)['prediction'],{**key,'predictor':'soft_segment_mixture'}))
            oracle_means={}
            for lab in ['S1','S2','S3']:
                means=[X[tr][:,eg][(specimens[tr]==s)&(labels[tr]==lab)].mean(axis=0) for s in np.unique(specimens[tr]) if np.any((specimens[tr]==s)&(labels[tr]==lab))]
                if not means:
                    raise ValueError('Missing training segment '+lab)
                oracle_means[lab]=np.mean(means,axis=0)
            scores.extend(score_rows(tr,te,eg,np.array([oracle_means[lab] for lab in labels[te]]),{**key,'predictor':'segment_oracle'}))
            diagnostics.append(dict(protocol=protocol,gaussian_converged=gaussian['converged'],minimum_per_specimen_support=float(rate['n_effective'].min()),rate_gene_count=len(mg),NB_mean_entropy=float(nb_query['entropy'].mean()),gaussian_mean_entropy=float(ga_query['entropy'].mean())))
            # Thinning is restricted to the joint interpolation query; transfer
            # is independently evaluated above, not counted as extra replicates.
            if protocol=='joint_profile_holdout':
                for fraction in [.75,.5,.25]:
                    for repeat in range(2):
                        thin=thin_counts(raw_counts[te],fraction,SEED+int(1000*fraction)+repeat)
                        exposure=np.asarray(thin.sum(axis=1)).ravel()
                        if np.any(exposure<=0):
                            failures.append(dict(protocol=protocol,error='Empty thinned query',fraction=fraction,repeat=repeat))
                            continue
                        panel=thin[:,mg]
                        panel=panel.toarray() if sparse.issparse(panel) else np.asarray(panel)
                        thinned={'count_gaussian':gaussian_projection(gaussian,transform_count_representation(panel,exposure,transform),specimens[te]),'count_NB':project_count_rate_atlas(rate,panel,exposure)}
                        original={'count_gaussian':ga_query,'count_NB':nb_query}
                        for method,pp in thinned.items():
                            base_projection=original[method]
                            for s in CONTROLS:
                                for segment in ['all','S1','S2','S3']:
                                    use=specimens[te]==s
                                    if segment!='all':
                                        use&=labels[te]==segment
                                    delta=np.abs(pp['z_mean'][use]-base_projection['z_mean'][use])
                                    thinning_rows.append(dict(protocol=protocol,method=method,fraction=fraction,repeat=repeat,specimen=s,segment=segment,n=int(use.sum()),mean_absolute_shift=float(delta.mean()),p90_absolute_shift=float(np.quantile(delta,.9)),fraction_shift_over_point1=float(np.mean(delta>.1)),rank_spearman=float(spearmanr(pp['z_mean'][use],base_projection['z_mean'][use]).statistic),mean_entropy_change=float(np.mean(pp['entropy'][use]-base_projection['entropy'][use]))))
                        print('NB same-section thinning:',fraction,repeat,flush=True)
            print('Rate pilot finished:',protocol,flush=True)
        except (ValueError,RuntimeError,np.linalg.LinAlgError) as exc:
            failures.append(dict(protocol=protocol,error=str(exc)))
            print('Rate projection failure:',failures[-1],flush=True)
    frames={'scores':pd.DataFrame(scores),'coordinates':pd.DataFrame(coordinates),'thinning':pd.DataFrame(thinning_rows),'diagnostics':pd.DataFrame(diagnostics),'failures':pd.DataFrame(failures)}
    return {k:v.to_json(orient='table') for k,v in frames.items()}
rate_payload=cached_payload('conditional_count_rate_projection',run_rate_projection,root=OUT/'stage_cache',params=CACHE_PARAMS,inputs=CACHE_INPUTS,code=CODE_DIGEST)
rate_outputs={k:pd.read_json(io.StringIO(str(v)),orient='table') for k,v in rate_payload.items()}
for name,frame in rate_outputs.items():
    frame.to_csv(OUT/(name+'.csv'),index=False)
display(rate_outputs['scores'].query("segment=='all'").pivot_table(index=['protocol','test_specimen'],columns='predictor',values='gene_mse'))
display(rate_outputs['diagnostics'])
display(rate_outputs['failures'])

## Robustness evidence and remaining interpretation limits

Count thinning changes read coverage while keeping the physical section fixed. Compare both point movements and within-segment rank stability; a global correlation can conceal rearrangements within S1/S2/S3. Entropy changes are conditional on the model, and low NB entropy does not establish confidence because genes are correlated and biological residual variation may exceed the fixed dispersion.

A better decoder does not prove a correct coordinate. This pilot inherits the training axis from the count-Gaussian reconstruction, estimates a frozen count-density path, and changes projection. A full joint count estimator would require additional model fitting and safeguards. AKI, human homology and TF interpretations remain future work.

In [ ]:
thinning_summary=rate_outputs['thinning']
display(thinning_summary.query("segment=='all'").groupby(['method','fraction','specimen'])[['mean_absolute_shift','p90_absolute_shift','fraction_shift_over_point1','rank_spearman','mean_entropy_change']].mean())
display(thinning_summary.query("segment!='all'").groupby(['method','fraction','segment'])[['mean_absolute_shift','rank_spearman']].mean())
fig,ax=plt.subplots(figsize=(6,3),layout='constrained')
for method,g in thinning_summary.query("segment=='all'").groupby('method'):
    table=g.groupby('fraction').mean(numeric_only=True)
    ax.plot(table.index,table.mean_absolute_shift,'o-',label=method)
ax.set(xlabel='Fraction of retained reads',ylabel='Mean absolute coordinate movement')
ax.legend(frameon=False)
fig.savefig(OUT/'same_section_thinning_comparison.png',dpi=160)
plt.show()